<a href="https://colab.research.google.com/github/kimdesok/V-JEPA/blob/main/VL_JEPA_Prototype.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install decord

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.6/13.6 MB 95.3 MB/s eta 0:00:00


In [ ]:
!git clone https://github.com/facebookresearch/vjepa2.git

Cloning into 'vjepa2'...
remote: Enumerating objects: 310, done.
remote: Counting objects: 100% (71/71), done.
remote: Compressing objects: 100% (38/38), done.
remote: Total 310 (delta 45), reused 33 (delta 33), pack-reused 239 (from 3)
Receiving objects: 100% (310/310), 576.64 KiB | 21.36 MiB/s, done.
Resolving deltas: 100% (141/141), done.


In [ ]:
cd vjepa2

/content/vjepa2


In [ ]:
!pip install -e .

Obtaining file:///content/vjepa2
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.2/42.2 kB 3.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.9/115.9 kB 11.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.0/76.0 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.0/75.0 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 98.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.7/76.7 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.8/59.8 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 85.5 MB/s eta 0:00:00
  Building 

In [ ]:
!pip uninstall -y torch torchvision torchaudio
!pip install -q torch==2.9.0 torchvision==0.24.0 torchaudio==2.9.0 --index-url https://download.pytorch.org/whl/cu126
!pip install -q --no-deps torchcodec xformers accelerate

Found existing installation: torch 2.9.0+cu126
Uninstalling torch-2.9.0+cu126:
  Successfully uninstalled torch-2.9.0+cu126
Found existing installation: torchvision 0.24.0+cu126
Uninstalling torchvision-0.24.0+cu126:
  Successfully uninstalled torchvision-0.24.0+cu126
Found existing installation: torchaudio 2.9.0+cu126
Uninstalling torchaudio-2.9.0+cu126:
  Successfully uninstalled torchaudio-2.9.0+cu126
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 832.9/832.9 MB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.3/7.3 MB 101.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 84.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 96.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 110.8/110.8 MB 23.0 MB/s eta 0:00:00


In [ ]:
import torch
from transformers import AutoTokenizer, CLIPTextModel

text_model_name = "openai/clip-vit-base-patch32"
tokenizer = AutoTokenizer.from_pretrained(text_model_name)
text_encoder = CLIPTextModel.from_pretrained(text_model_name).cuda().eval()

@torch.no_grad()
def encode_text(texts):  # list[str]
    toks = tokenizer(texts, padding=True, truncation=True, return_tensors="pt").to("cuda")
    out = text_encoder(**toks)
    # CLIPTextModel returns last_hidden_state; use EOS token embedding or mean-pool
    z = out.last_hidden_state.mean(dim=1)   # (B, D_text)
    z = F.normalize(z, dim=-1)
    return z


In [ ]:
def pool_video_tokens(z_tokens):  # (B, N, D)
    z = z_tokens.mean(dim=1)      # (B, D)
    z = F.normalize(z, dim=-1)
    return z

In [ ]:
import torch.nn as nn

class VideoToTextPredictor(nn.Module):
    def __init__(self, d_video, d_text, hidden=2048):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_video, hidden),
            nn.GELU(),
            nn.Linear(hidden, d_text),
        )
    def forward(self, z_v):
        z_t_hat = self.net(z_v)
        z_t_hat = F.normalize(z_t_hat, dim=-1)
        return z_t_hat

def cosine_loss(z_hat, z):
    return 1.0 - (z_hat * z).sum(dim=-1).mean()

def make_text_from_ssv2_label(label: str) -> str:
    # Keep it simple first; later add multiple templates per label
    return f"A video of {label.lower()}."

In [ ]:
import json
import os
import subprocess

import numpy as np
import torch
import torch.nn.functional as F
from decord import VideoReader
from transformers import AutoVideoProcessor, AutoModel

import src.datasets.utils.video.transforms as video_transforms
import src.datasets.utils.video.volume_transforms as volume_transforms
from src.models.attentive_pooler import AttentiveClassifier
from src.models.vision_transformer import vit_giant_xformers_rope

IMAGENET_DEFAULT_MEAN = (0.485, 0.456, 0.406)
IMAGENET_DEFAULT_STD = (0.229, 0.224, 0.225)

def load_pretrained_vjepa_pt_weights(model, pretrained_weights):
    # Load weights of the VJEPA2 encoder
    # The PyTorch state_dict is already preprocessed to have the right key names
    pretrained_dict = torch.load(pretrained_weights, weights_only=True, map_location="cpu")["encoder"]
    pretrained_dict = {k.replace("module.", ""): v for k, v in pretrained_dict.items()}
    pretrained_dict = {k.replace("backbone.", ""): v for k, v in pretrained_dict.items()}
    msg = model.load_state_dict(pretrained_dict, strict=False)
    print("Pretrained weights found at {} and loaded with msg: {}".format(pretrained_weights, msg))


def load_pretrained_vjepa_classifier_weights(model, pretrained_weights):
    # Load weights of the VJEPA2 classifier
    # The PyTorch state_dict is already preprocessed to have the right key names
    pretrained_dict = torch.load(pretrained_weights, weights_only=True, map_location="cpu")["classifiers"][0]
    pretrained_dict = {k.replace("module.", ""): v for k, v in pretrained_dict.items()}
    msg = model.load_state_dict(pretrained_dict, strict=False)
    print("Pretrained weights found at {} and loaded with msg: {}".format(pretrained_weights, msg))


def build_pt_video_transform(img_size):
    short_side_size = int(256.0 / 224 * img_size)
    # Eval transform has no random cropping nor flip
    eval_transform = video_transforms.Compose(
        [
            video_transforms.Resize(short_side_size, interpolation="bilinear"),
            video_transforms.CenterCrop(size=(img_size, img_size)),
            volume_transforms.ClipToTensor(),
            video_transforms.Normalize(mean=IMAGENET_DEFAULT_MEAN, std=IMAGENET_DEFAULT_STD),
        ]
    )
    return eval_transform


def get_video():
    vr = VideoReader("sample_video.mp4")
    # choosing some frames here, you can define more complex sampling strategy
    frame_idx = np.arange(0, 128, 2)
    video = vr.get_batch(frame_idx).asnumpy()
    return video


def forward_vjepa_video(model_hf, model_pt, hf_transform, pt_transform):
    # Run a sample inference with VJEPA
    with torch.inference_mode():
        # Read and pre-process the image
        video = get_video()  # T x H x W x C
        video = torch.from_numpy(video).permute(0, 3, 1, 2)  # T x C x H x W
        x_pt = pt_transform(video).cuda().unsqueeze(0)
        x_hf = hf_transform(video, return_tensors="pt")["pixel_values_videos"].to("cuda")
        # Extract the patch-wise features from the last layer
        out_patch_features_pt = model_pt(x_pt)
        out_patch_features_hf = model_hf.get_vision_features(x_hf)

    return out_patch_features_hf, out_patch_features_pt


def get_vjepa_video_classification_results(classifier, out_patch_features_pt):
    SOMETHING_SOMETHING_V2_CLASSES = json.load(open("ssv2_classes.json", "r"))

    with torch.inference_mode():
        out_classifier = classifier(out_patch_features_pt)

    print(f"Classifier output shape: {out_classifier.shape}")

    print("Top 5 predicted class names:")
    top5_indices = out_classifier.topk(5).indices[0]
    top5_probs = F.softmax(out_classifier.topk(5).values[0]) * 100.0  # convert to percentage
    for idx, prob in zip(top5_indices, top5_probs):
        str_idx = str(idx.item())
        print(f"{SOMETHING_SOMETHING_V2_CLASSES[str_idx]} ({prob}%)")

    return

In [ ]:
sample_video_path = "sample_video.mp4"
# Download the video if not yet downloaded to local path
if not os.path.exists(sample_video_path):
    video_url = "https://huggingface.co/datasets/nateraw/kinetics-mini/resolve/main/val/bowling/-WH-lxmGJVY_000005_000015.mp4"
    command = ["wget", video_url, "-O", sample_video_path]
    subprocess.run(command)
    print("Downloading video")

# Download SSV2 classes if not already present
ssv2_classes_path = "ssv2_classes.json"
if not os.path.exists(ssv2_classes_path):
    command = [
        "wget",
        "https://huggingface.co/datasets/huggingface/label-files/resolve/d79675f2d50a7b1ecf98923d42c30526a51818e2/"
        "something-something-v2-id2label.json",
        "-O",
        "ssv2_classes.json",
    ]
    subprocess.run(command)
    print("Downloading SSV2 classes")

In [ ]:
from IPython.display import HTML
from base64 import b64encode

def play_video(video_path, width=600):
    """Encodes and displays a local .mp4 video file in Colab."""
    video_file = open(video_path, "rb").read()
    # Create a data URL for the video
    video_url = f"data:video/mp4;base64,{b64encode(video_file).decode()}"
    return HTML(f"""
        <video width={width} controls>
            <source src="{video_url}" type="video/mp4">
        </video>
    """)

# Replace 'sample_video.mp4' with your actual file path
play_video('sample_video.mp4')

In [ ]:
!wget https://dl.fbaipublicfiles.com/vjepa2/vitg-384.pt -P ./weights

--2026-01-27 10:06:17--  https://dl.fbaipublicfiles.com/vjepa2/vitg-384.pt
Resolving dl.fbaipublicfiles.com (dl.fbaipublicfiles.com)... 13.35.238.103, 13.35.238.71, 13.35.238.84, ...
Connecting to dl.fbaipublicfiles.com (dl.fbaipublicfiles.com)|13.35.238.103|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 16464301382 (15G) [application/xml]
Saving to: ‘./weights/vitg-384.pt’

vitg-384.pt         100%[===================>]  15.33G   310MB/s    in 58s     

2026-01-27 10:07:15 (272 MB/s) - ‘./weights/vitg-384.pt’ saved [16464301382/16464301382]



In [ ]:
# HuggingFace model repo name
hf_model_name = (
    "facebook/vjepa2-vitg-fpc64-384"  # Replace with your favored model, e.g. facebook/vjepa2-vitg-fpc64-384
)
# Path to local PyTorch weights
pt_model_path = "./weights/vitg-384.pt"

# Initialize the HuggingFace model, load pretrained weights
model_hf = AutoModel.from_pretrained(hf_model_name)
model_hf.cuda().eval()

# Build HuggingFace preprocessing transform
hf_transform = AutoVideoProcessor.from_pretrained(hf_model_name)
img_size = hf_transform.crop_size["height"]  # E.g. 384, 256, etc.

# Initialize the PyTorch model, load pretrained weights
model_pt = vit_giant_xformers_rope(img_size=(img_size, img_size), num_frames=64)
model_pt.cuda().eval()

### Can also use torch.hub to load the model
#model_pt, _ = torch.hub.load('facebookresearch/vjepa2', 'vjepa2_vit_giant_384')
#model_pt.cuda().eval()

load_pretrained_vjepa_pt_weights(model_pt, pt_model_path)

# Build PyTorch preprocessing transform
pt_video_transform = build_pt_video_transform(img_size=img_size)

Pretrained weights found at ./weights/vitg-384.pt and loaded with msg: <All keys matched successfully>


In [ ]:
# Inference on video to get the patch-wise features
out_patch_features_hf, out_patch_features_pt = forward_vjepa_video(
    model_hf, model_pt, hf_transform, pt_video_transform
)

print(
    f"""
    Inference results on video:
    HuggingFace output shape: {out_patch_features_hf.shape}
    PyTorch output shape:     {out_patch_features_pt.shape}
    Absolute difference sum:  {torch.abs(out_patch_features_pt - out_patch_features_hf).sum():.6f}
    Close: {torch.allclose(out_patch_features_pt, out_patch_features_hf, atol=1e-3, rtol=1e-3)}
    """
)


    Inference results on video:
    HuggingFace output shape: torch.Size([1, 18432, 1408])
    PyTorch output shape:     torch.Size([1, 18432, 1408])
    Absolute difference sum:  10306020.000000
    Close: False
    


In [ ]:
!wget https://dl.fbaipublicfiles.com/vjepa2/evals/ssv2-vitg-384-64x2x3.pt -P ./weights

--2026-01-27 11:51:36--  https://dl.fbaipublicfiles.com/vjepa2/evals/ssv2-vitg-384-64x2x3.pt
Resolving dl.fbaipublicfiles.com (dl.fbaipublicfiles.com)... 65.8.76.77, 65.8.76.89, 65.8.76.47, ...
Connecting to dl.fbaipublicfiles.com (dl.fbaipublicfiles.com)|65.8.76.77|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 373996872 (357M) [application/vnd.snesdev-page-table]
Saving to: ‘./weights/ssv2-vitg-384-64x2x3.pt.1’

ssv2-vitg-384-64x2x 100%[===================>] 356.67M   141MB/s    in 2.5s    

2026-01-27 11:51:38 (141 MB/s) - ‘./weights/ssv2-vitg-384-64x2x3.pt.1’ saved [373996872/373996872]



In [ ]:
# Initialize the classifier
classifier_model_path = "./weights/ssv2-vitg-384-64x2x3.pt"
classifier = (
    AttentiveClassifier(embed_dim=model_pt.embed_dim, num_heads=16, depth=4, num_classes=174).cuda().eval()
)
load_pretrained_vjepa_classifier_weights(classifier, classifier_model_path)

# Get classification results
get_vjepa_video_classification_results(classifier, out_patch_features_pt)

Pretrained weights found at ./weights/ssv2-vitg-384-64x2x3.pt and loaded with msg: <All keys matched successfully>
Classifier output shape: torch.Size([1, 174])
Top 5 predicted class names:


/tmp/ipython-input-2987202713.py:85: UserWarning: Implicit dimension choice for softmax has been deprecated. Change the call to include dim=X as an argument.
  top5_probs = F.softmax(out_classifier.topk(5).values[0]) * 100.0  # convert to percentage


Putting [something] into [something] (45.151634216308594%)
Stuffing [something] into [something] (27.54499053955078%)
Putting [something] onto [something] (15.023070335388184%)
Failing to put [something] into [something] because [something] does not fit (7.21915864944458%)
Putting [number of] [something] onto [something] (5.061150550842285%)


In [ ]:
import os, json
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from decord import VideoReader

class VideoTextDataset(Dataset):
    """
    Expects an annotation list of dicts like:
    [
      {"video": "path/to/vid.mp4", "label": "Taking [something] from [something]"},
      ...
    ]
    """
    def __init__(
        self,
        ann_path: str,
        video_root: str = "",
        pt_transform=None,
        num_frames: int = 64,
        stride: int = 2,
        start: int = 0,
        text_template: str = "A video of {label}."
    ):
        self.video_root = video_root
        self.pt_transform = pt_transform
        self.num_frames = num_frames
        self.stride = stride
        self.start = start
        self.text_template = text_template

        with open(ann_path, "r") as f:
            self.items = json.load(f)

        if not isinstance(self.items, list) or len(self.items) == 0:
            raise ValueError("Annotation file must be a non-empty list of dicts.")

    def __len__(self):
        return len(self.items)

    def _sample_indices(self, n_total: int):
        # fixed strategy: start + arange(num_frames)*stride
        idx = self.start + np.arange(self.num_frames) * self.stride
        # clamp to valid range
        idx = np.clip(idx, 0, n_total - 1)
        return idx.astype(np.int64)

    def __getitem__(self, i: int):
        it = self.items[i]
        rel_path = it["video"]
        label = it.get("label", "")
        video_path = rel_path if os.path.isabs(rel_path) else os.path.join(self.video_root, rel_path)

        vr = VideoReader(video_path)
        n_total = len(vr)

        frame_idx = self._sample_indices(n_total)
        video = vr.get_batch(frame_idx).asnumpy()              # (T, H, W, C) uint8
        video = torch.from_numpy(video).permute(0, 3, 1, 2)    # (T, C, H, W)

        # Your pt_transform expects (T, C, H, W) and returns (C, T, H, W)
        x_pt = self.pt_transform(video)                        # (C, T, H, W), float normalized

        text = self.text_template.format(label=str(label).lower())

        return x_pt, text



In [ ]:
import json

toy = [
    {"video": "sample_video.mp4", "label": "Bowling a ball"},
    {"video": "sample_video.mp4", "label": "Rolling something on a surface"},
]
with open("train_vl.json", "w") as f:
    json.dump(toy, f, indent=2)


In [ ]:
def collate_video_text(batch):
    xs, texts = zip(*batch)
    x = torch.stack(xs, dim=0)   # (B, C, T, H, W)
    return x, list(texts)


In [ ]:
ann_path = "train_vl.json"   # you create this (see below)
video_root = "/content/vjepa2"             # wherever the mp4s live

dataset = VideoTextDataset(
    ann_path=ann_path,
    video_root=video_root,
    pt_transform=pt_video_transform,
    num_frames=64,     # matches your vjepa2 config
    stride=2,
    start=0,
    text_template="A video of {label}."
)

loader = DataLoader(
    dataset,
    batch_size=2,
    shuffle=True,
    num_workers=0,
    pin_memory=True,
    collate_fn=collate_video_text,
    drop_last=False
)


In [ ]:
predictor = VideoToTextPredictor(
    d_video=model_pt.embed_dim,
    d_text=text_encoder.config.hidden_size
).cuda()

opt = torch.optim.AdamW(predictor.parameters(), lr=1e-4, weight_decay=0.05)

model_pt.eval()
text_encoder.eval()

for step, batch in enumerate(loader):
    #print(step)
    x_pt, labels = batch  # you define this
    x_pt = x_pt.cuda()

    with torch.no_grad():
        z_tokens = model_pt(x_pt)          # (B, N, D_video)
        z_v = pool_video_tokens(z_tokens).detach()  # (B, D_video)

        texts = [make_text_from_ssv2_label(lbl) for lbl in labels]
        z_t = encode_text(texts).detach()         # (B, D_text)

    z_t_hat = predictor(z_v)
    loss = cosine_loss(z_t_hat, z_t)

    opt.zero_grad(set_to_none=True)
    loss.backward()
    opt.step()

    if step % 50 == 0:
        print(step, float(loss))

0 0.973958432674408


/tmp/ipython-input-2910182340.py:31: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:836.)
  print(step, float(loss))


In [ ]:
x, texts = next(iter(loader))
print(x.shape)     # (B, C, T, H, W)
print(texts[:2])


torch.Size([2, 3, 64, 384, 384])
['A video of rolling something on a surface.', 'A video of bowling a ball.']


In [ ]:
@torch.inference_mode()
def score_texts_for_video(x_pt, candidate_texts):
    z_tokens = model_pt(x_pt.cuda())
    z_v = pool_video_tokens(z_tokens)
    z_t_hat = predictor(z_v)                 # predicted text embedding

    z_cand = encode_text(candidate_texts)    # (K, D_text)
    sims = (z_t_hat @ z_cand.T).squeeze(0)   # (K,)
    return sims


In [ ]:
for x_pt, texts in loader:
    x_pt = x_pt.cuda(non_blocking=True)

    with torch.no_grad():
        z_tokens = model_pt(x_pt)            # (B, N, D)
        z_v = z_tokens.mean(dim=1)           # (B, D)
        z_v = F.normalize(z_v, dim=-1)

        z_t = encode_text(texts)             # (B, D_text)

    z_t_hat = predictor(z_v)
    loss = 1.0 - (z_t_hat * z_t).sum(dim=-1).mean()

    opt.zero_grad(set_to_none=True)
    loss.backward()
    opt.step()
